# Statistical Analysis 

In [6]:
import pandas as pd 
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

This line imports utils:

In [7]:
%run ../../utils/utils.py

## 1. Load Data

In [8]:
df = pd.read_csv("../../data/preprocessed.csv")
df = data_featuring_sensitivity(df)
df.head()

,s1_edad,s1_genero,s1_experiencia_clinica,s1_educacion,s1_titulo_terciario,s1_titulo,s1_no_titulo_universitario,s1_horas_semana_pacientes_atendidos,s1_contexto_trabajo,s1_orientacion_teo,...,s4_no_tiempo_aprender_tbe,s4_capacitacion_tbe_demasiado_dinero,s4_no_saber_tbe,s4_entrenamiento_clinico_no_info_tbe,s4_alianza_terapeutica_mas_importante,s4_terapias_igualmente_efectivas,s4_empleador_no_fondos_capacitacion_tbe,s4_exp_clinica_mas_importante_que_evidencia_cientifica,provincia_residencia,consentimiento_informado
0,41.0,Femenino,1.2,Licenciatura de Grado,NaN,Lic. en Psicología,NaN,12.0,Ámbito Privado,Ecléctico (más de una de estas opciones),...,4.0,4.0,7.0,NaN,6.0,4.0,6.0,2.0,Provincia de Buenos Aires,NaN
1,26.0,Femenino,0.2,Licenciatura de Grado,NaN,Lic. en Psicología,NaN,13.0,"Ámbito Privado, Obra Social o Prepaga",Terapias Cognitivas/Comportamentales,...,1.0,6.0,1.0,NaN,4.0,2.0,7.0,3.0,Provincia de Buenos Aires,NaN
2,27.0,Femenino,1.0,Carrera de Especialización,NaN,Lic. en Psicología,NaN,30.0,"Ámbito Privado, Obra Social o Prepaga",Terapias Cognitivas/Comportamentales,...,1.0,4.0,1.0,1.0,1.0,1.0,NaN,4.0,Ciudad Autónoma de Buenos Aires (CABA),NaN
3,30.0,Masculino,1.3,Carrera de Especialización,NaN,Lic. en Psicología,NaN,35.0,Ámbito Privado,Terapias Cognitivas/Comportamentales,...,2.0,3.0,NaN,4.0,3.0,NaN,7.0,3.0,Provincia de Buenos Aires,NaN
4,26.0,Masculino,2.0,Licenciatura de Grado,NaN,Lic. en Psicología,NaN,46.0,Ámbito Público (hospital u otro),Terapias Cognitivas/Comportamentales,...,4.0,6.0,1.0,4.0,4.0,1.0,7.0,4.0,Provincia de Buenos Aires,NaN


In [9]:
"""
=============================================================
ANALYSIS 1: Variable-by-variable ANOVA (zeros excluded)
ANALYSIS 2: Proportion of zeros per variable per orientation
=============================================================

Instructions:
- Paste this code into a new notebook cell after your existing setup cells
- Requires: df already loaded (sensitivity/clean version, zeros → NaN)
- featured_data_cluster.csv must exist (original, zeros kept)
=============================================================
"""

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.stats.multicomp as multi
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────
subset_orientations = [
    'Ecléctico (más de una de estas opciones)',
    'Terapias Cognitivas/Comportamentales',
    'Psicoanálisis',
    'Sistémica'
]

variables_s2 = [
    's2_evidencia_cientifica', 's2_experiencia_personal',
    's2_entrenamiento_clinica', 's2_tratamiento_preferencia_consultantes',
    's2_intuicion', 's2_terapia_personal'
]

variables_s3 = [
    's3_tratamiento_personal_consultantes',
    's3_investigacion_empirica_ensayos_controlados',
    's3_supervision', 's3_estudios_de_caso', 's3_discusion_pares',
    's3_libros', 's3_observaciones_casos_clinicos',
    's3_medidas_resultado', 's3_guias_manuales_clinicos'
]

variables_s4 = [
    's4_apertura_terapias_desarrolladas_por_investigadores',
    's4_nueva_terapia_intento',
    's4_terapia_manualizada', 's4_diagnosticos_utilizados_son_simples',
    's4_tratamientos_preferencia_no_probados_ensayo_controlado',
    's4_enfoque_tratamiento_individual',
    's4_alianza_terapeutica_mas_importante',
    's4_terapias_igualmente_efectivas',
    's4_exp_clinica_mas_importante_que_evidencia_cientifica',
    's4_actualizacion_info_cientifica',
    's4_formacion_enfasis_investigacion',
    's4_supervisores_terapia_evidencia_requerimiento',
    's4_atraer_consultantes_con_tbe',
    's4_hallazgos_cientificos_practica_diaria',
    's4_interes_aprender_tbe',
    's4_tratamientos_utilizados_base_empirica',
    's4_complejidad_consultantes_ensayos_clinicos',
    's4_consultantes_prefieren_otros_tratamientos',
    's4_no_tiempo_aprender_tbe',
    's4_capacitacion_tbe_demasiado_dinero',
    's4_no_saber_tbe',
    's4_entrenamiento_clinico_no_info_tbe',
    's4_empleador_no_fondos_capacitacion_tbe'
]


# ── HELPER FUNCTIONS ──────────────────────────────────────

def eta_squared(groups):
    """Calculate eta squared from list of group arrays."""
    all_data = np.concatenate(groups)
    grand_mean = np.mean(all_data)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = sum((x - grand_mean)**2 for x in all_data)
    return round(ss_between / ss_total, 3) if ss_total > 0 else np.nan

def interpret_eta(eta2):
    if np.isnan(eta2): return 'N/A'
    if eta2 >= 0.14: return 'Large'
    if eta2 >= 0.06: return 'Medium'
    if eta2 >= 0.01: return 'Small'
    return 'Negligible'

def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


# ══════════════════════════════════════════════════════════
# ANALYSIS 1: ANOVA variable-by-variable (zeros excluded)
# Each variable uses its own available N after removing zeros
# ══════════════════════════════════════════════════════════

def run_anova_per_variable(df, variables, section_name, exclude_zeros=True):
    """
    Run one-way ANOVA per variable with optional zero exclusion.
    Followed by Tukey HSD for significant variables.

    Parameters:
    -----------
    df : DataFrame with s1_orientacion_teo column
    variables : list of variable names
    section_name : string label for printing
    exclude_zeros : if True, removes 0s before analysis (treats as missing)
    """
    df_sub = df[df['s1_orientacion_teo'].isin(subset_orientations)].copy()

    print(f"\n{'='*90}")
    print(f"  ANOVA — {section_name} ({'zeros excluded' if exclude_zeros else 'zeros included'})")
    print(f"{'='*90}")
    print(f"\n{'Variable':<52} {'N':>5} {'F':>8} {'p':>8} {'η²':>6} {'Size':>9} {'sig':>5}")
    print("-"*92)

    significant_vars = []
    results = []

    for v in variables:
        if v not in df_sub.columns:
            continue

        groups = []
        for o in subset_orientations:
            grp = df_sub[df_sub['s1_orientacion_teo'] == o][v].dropna()
            if exclude_zeros:
                grp = grp[grp != 0]
            groups.append(grp.values)

        groups_valid = [g for g in groups if len(g) >= 2]
        if len(groups_valid) < 2:
            continue

        total_n = sum(len(g) for g in groups_valid)
        f, p = stats.f_oneway(*groups_valid)
        eta2 = eta_squared(groups_valid)
        size = interpret_eta(eta2)
        sig = sig_label(p)

        print(f"{v:<52} {total_n:>5} {f:>8.3f} {p:>8.4f} {eta2:>6.3f} {size:>9} {sig:>5}")

        results.append({
            'variable': v, 'n': total_n, 'F': round(f, 3),
            'p': round(p, 4), 'eta2': eta2, 'size': size, 'sig': sig
        })

        if p < 0.05:
            significant_vars.append(v)

    # ── Tukey HSD for significant variables ──
    print(f"\n\n{'─'*90}")
    print(f"  TUKEY HSD — significant variables only (p < .05)")
    print(f"{'─'*90}")

    for v in significant_vars:
        grp_data = []
        grp_labels = []
        for o in subset_orientations:
            grp = df_sub[df_sub['s1_orientacion_teo'] == o][v].dropna()
            if exclude_zeros:
                grp = grp[grp != 0]
            grp_data.extend(grp.tolist())
            grp_labels.extend([o] * len(grp))

        if len(set(grp_labels)) < 2:
            continue

        mc = multi.MultiComparison(
            pd.Series(grp_data),
            pd.Series(grp_labels)
        )
        res = mc.tukeyhsd()
        print(f"\n{v}")
        print(res.summary())

    return pd.DataFrame(results)


# ══════════════════════════════════════════════════════════
# ANALYSIS 2: Proportion of zeros per variable per orientation
# Chi-square test: are zeros distributed differently across orientations?
# ══════════════════════════════════════════════════════════

def run_proportion_analysis(df, variables, section_name, count_nan=False):
    """
    For each variable: count 0s (or NaNs) per orientation group.
    Chi-square test on whether distribution of 0s differs across orientations.

    Parameters:
    -----------
    df : DataFrame
    variables : list of variable names
    section_name : string label
    count_nan : if True, count NaN instead of 0 (for S2 where 0 was recoded to NaN)
    """
    df_sub = df[df['s1_orientacion_teo'].isin(subset_orientations)].copy()

    short = {
        'Ecléctico (más de una de estas opciones)': 'Ecl',
        'Terapias Cognitivas/Comportamentales':     'TCC',
        'Psicoanálisis':                             'PSA',
        'Sistémica':                                 'Sis'
    }

    print(f"\n{'='*105}")
    print(f"  PROPORTION OF {'NaN' if count_nan else '0'} RESPONSES — {section_name}")
    print(f"{'='*105}")
    print(f"\n{'Variable':<50} {'TCC':>10} {'PSA':>14} {'Ecl':>10} {'Sis':>10} "
          f"{'Total':>10} {'χ²':>7} {'p':>8} {'sig':>5}")
    print("-"*107)

    results = []

    for v in variables:
        if v not in df_sub.columns:
            continue

        contingency = []
        cell_values = {}

        for o, s in short.items():
            if count_nan:
                grp_all = df_sub[df_sub['s1_orientacion_teo'] == o][v]
                n_total = len(grp_all)
                n_zero = int(grp_all.isna().sum())
            else:
                grp_all = df_sub[df_sub['s1_orientacion_teo'] == o][v].dropna()
                n_total = len(grp_all)
                n_zero = int((grp_all == 0).sum())

            n_nonzero = n_total - n_zero
            pct = round(100 * n_zero / n_total, 1) if n_total > 0 else 0

            cell_values[s] = f"{n_zero}/{n_total} ({pct}%)"
            contingency.append([n_zero, n_nonzero])

        # Total
        ct = np.array(contingency)
        total_zero = int(ct[:, 0].sum())
        total_n = int(ct.sum())
        total_pct = round(100 * total_zero / total_n, 1) if total_n > 0 else 0

        # Chi-square
        try:
            chi2_val, p_val, dof, expected = chi2_contingency(ct)
            low_expected = (expected < 5).any()
            note = '~' if low_expected else ''
            sig = sig_label(p_val)
        except Exception:
            chi2_val, p_val, sig, note = np.nan, np.nan, 'err', ''

        print(f"{v:<50} {cell_values['TCC']:>10} {cell_values['PSA']:>14} "
              f"{cell_values['Ecl']:>10} {cell_values['Sis']:>10} "
              f"{total_zero}/{total_n}({total_pct}%){note:>1} "
              f"{chi2_val:>6.2f} {p_val:>8.4f} {sig:>5}")

        # Extract percentages safely
        def extract_pct(s):
            try:
                return float(s.split('(')[1].rstrip('%)').strip())
            except Exception:
                return 0.0

        results.append({
            'variable': v,
            'TCC_pct': extract_pct(cell_values['TCC']),
            'PSA_pct': extract_pct(cell_values['PSA']),
            'Ecl_pct': extract_pct(cell_values['Ecl']),
            'Sis_pct': extract_pct(cell_values['Sis']),
            'total_pct': total_pct,
            'chi2': round(chi2_val, 3) if not np.isnan(chi2_val) else np.nan,
            'p': round(p_val, 4) if not np.isnan(p_val) else np.nan,
            'sig': sig,
            'low_expected': note == '~'
        })

    # Summary of significant variables
    sig_results = [r for r in results if r.get('p', 1) < 0.05]
    print(f"\n  Significant χ² (p < .05): {len(sig_results)}/{len(results)} variables")
    if sig_results:
        print(f"\n  {'Variable':<50} {'χ²':>7} {'p':>8} {'sig':>5} {'PSA%':>6} {'TCC%':>6}")
        print(f"  {'-'*85}")
        for r in sorted(sig_results, key=lambda x: x['p']):
            print(f"  {r['variable']:<50} {r['chi2']:>7.3f} {r['p']:>8.4f} "
                  f"{r['sig']:>5} {r['PSA_pct']:>5.1f}% {r['TCC_pct']:>5.1f}%")

    print(f"\n  Note: ~ indicates expected cell frequencies < 5 (interpret with caution)")

    return pd.DataFrame(results)


# ══════════════════════════════════════════════════════════
# RUN ANALYSIS 1: ANOVA (on clean data — zeros already NaN)
# ══════════════════════════════════════════════════════════

print("\n\n" + "█"*90)
print("  RUNNING ANOVA WITH ZEROS EXCLUDED")
print("█"*90)

# df here is your sensitivity/clean dataframe (zeros recoded to NaN)
df_s2 = df[['s1_orientacion_teo'] + variables_s2].copy()
results_s2 = run_anova_per_variable(
    df_s2, variables_s2,
    'Section 2 — Factors influencing theoretical orientation',
    exclude_zeros=False  # S2 zeros already NaN in clean data
)

df_s3 = df[['s1_orientacion_teo'] + variables_s3].copy()
results_s3 = run_anova_per_variable(
    df_s3, variables_s3,
    'Section 3 — Sources used to improve clinical skills',
    exclude_zeros=True  # Remove any remaining 0s
)

df_s4 = df[['s1_orientacion_teo'] + variables_s4].copy()
results_s4 = run_anova_per_variable(
    df_s4, variables_s4,
    'Section 4 — Research attitudes',
    exclude_zeros=True  # Remove any remaining 0s
)


# ══════════════════════════════════════════════════════════
# RUN ANALYSIS 2: PROPORTION (on ORIGINAL data — zeros kept)
# Must load featured_data_cluster.csv which has zeros intact
# ══════════════════════════════════════════════════════════

print("\n\n" + "█"*90)
print("  RUNNING PROPORTION OF ZERO RESPONSES ANALYSIS")
print("  (using featured_data_cluster.csv — zeros kept as-is)")
print("█"*90)

# Load original data where zeros are still present
df_original = pd.read_csv('../../data/featured_data_cluster.csv')

df_s2_orig = df_original[['s1_orientacion_teo'] + variables_s2].copy()
df_s3_orig = df_original[['s1_orientacion_teo'] + variables_s3].copy()
df_s4_orig = df_original[['s1_orientacion_teo'] + variables_s4].copy()

prop_s2 = run_proportion_analysis(
    df_s2_orig, variables_s2,
    'Section 2 — Factors influencing theoretical orientation',
    count_nan=True   # S2 zeros are NaN even in original data
)

prop_s3 = run_proportion_analysis(
    df_s3_orig, variables_s3,
    'Section 3 — Sources used to improve clinical skills',
    count_nan=False  # zeros still present as 0
)

prop_s4 = run_proportion_analysis(
    df_s4_orig, variables_s4,
    'Section 4 — Research attitudes',
    count_nan=False  # zeros still present as 0
)


# ══════════════════════════════════════════════════════════
# SUMMARY TABLE
# ══════════════════════════════════════════════════════════

print("\n\n" + "█"*90)
print("  SUMMARY: AVERAGE % OF ZERO/NaN RESPONSES BY SECTION AND ORIENTATION")
print("█"*90)
print(f"\n{'Section':<15} {'N vars':>7} {'TCC%':>8} {'PSA%':>10} {'Ecl%':>8} {'Sis%':>8} {'Total%':>8}")
print("-"*65)
for label, prop_df in [('Section 2', prop_s2), ('Section 3', prop_s3), ('Section 4', prop_s4)]:
    print(f"{label:<15} {len(prop_df):>7} "
          f"{prop_df['TCC_pct'].mean():>7.1f}% "
          f"{prop_df['PSA_pct'].mean():>9.1f}% "
          f"{prop_df['Ecl_pct'].mean():>7.1f}% "
          f"{prop_df['Sis_pct'].mean():>7.1f}% "
          f"{prop_df['total_pct'].mean():>7.1f}%")



██████████████████████████████████████████████████████████████████████████████████████████
  RUNNING ANOVA WITH ZEROS EXCLUDED
██████████████████████████████████████████████████████████████████████████████████████████

  ANOVA — Section 2 — Factors influencing theoretical orientation (zeros included)

Variable                                                 N        F        p     η²      Size   sig
--------------------------------------------------------------------------------------------
s2_evidencia_cientifica                                256   51.541   0.0000  0.380     Large   ***
s2_experiencia_personal                                252    2.393   0.0690  0.028     Small    ns
s2_entrenamiento_clinica                               253    3.455   0.0171  0.040     Small     *
s2_tratamiento_preferencia_consultantes                233    1.488   0.2186  0.019     Small    ns
s2_intuicion                                           213    1.728   0.1623  0.024     Small    ns
s2

In [41]:
import numpy as np
import scipy.stats as stats
import statsmodels.stats.multicomp as mc
import statsmodels.stats.multitest as smt

def run_anova_per_variable_corrected(df, variables, section_name, exclude_zeros=True, alpha=0.05):
    """
    Ejecuta un ANOVA unidireccional por variable con exclusión de ceros y corrección FDR.
    Solo incluye 4 orientaciones teóricas específicas.
    Seguido de Tukey HSD solo para las variables significativas.
    """
    
    # Definir los 4 grupos que queremos analizar
    subset_orientations = [
        'Ecléctico (más de una de estas opciones)',
        'Terapias Cognitivas/Comportamentales',
        'Psicoanálisis',
        'Sistémica'
    ]
    
    print(f"\n{'='*105}")
    print(f"  ANOVA & FDR CORRECTION — {section_name} ({'ceros excluidos' if exclude_zeros else 'ceros incluidos'})")
    print(f"{'='*105}")
    
    # Almacenar resultados preliminares
    raw_results = []
    
    for v in variables:
        # Crear un subconjunto solo con la variable dependiente y la independiente
        subset = df[['s1_orientacion_teo', v]].copy()
        
        # 🔴 FILTRO CRÍTICO: Mantener solo las 4 orientaciones deseadas
        subset = subset[subset['s1_orientacion_teo'].isin(subset_orientations)]
        
        # Excluir ceros/nulos dinámicamente para esta variable específica
        if exclude_zeros:
            subset = subset[subset[v] != 0]
            
        subset = subset.dropna()
        
        # Agrupar los datos
        groups = [group[v].values for name, group in subset.groupby('s1_orientacion_teo')]
        total_n = sum(len(g) for g in groups)
        
        # Comprobar si tenemos suficientes grupos para ejecutar un ANOVA
        if len(groups) < 2 or any(len(g) == 0 for g in groups):
            raw_results.append({'var': v, 'n': total_n, 'f': np.nan, 'p_raw': np.nan, 'eta2': np.nan, 'groups': groups, 'subset': subset})
            continue
            
        # Ejecutar ANOVA estándar
        f_val, p_val = stats.f_oneway(*groups)
        
        # Calcular Eta-Cuadrado (Tamaño del efecto)
        all_data = np.concatenate(groups)
        grand_mean = np.mean(all_data)
        ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
        ss_total = sum((x - grand_mean)**2 for x in all_data)
        eta2 = ss_between / ss_total if ss_total > 0 else np.nan
        
        raw_results.append({
            'var': v, 
            'n': total_n, 
            'f': f_val, 
            'p_raw': p_val, 
            'eta2': eta2,
            'groups': groups,
            'subset': subset
        })

    # Filtrar las variables que no se pudieron calcular antes de la corrección
    valid_results = [r for r in raw_results if not np.isnan(r['p_raw'])]
    p_values = [r['p_raw'] for r in valid_results]
    
    # APLICAR CORRECCIÓN FDR DE BENJAMINI-HOCHBERG
    reject, pvals_corrected, _, _ = smt.multipletests(p_values, alpha=alpha, method='fdr_bh')
    
    # Imprimir la tabla de resumen
    print(f"\n{'Variable':<52} {'N':>5} {'F':>8} {'Raw p':>8} {'Adj p':>8} {'sig':>5}")
    print("-" * 95)
    
    significant_vars = []
    
    for i, r in enumerate(valid_results):
        v = r['var']
        n = r['n']
        f = r['f']
        p_raw = r['p_raw']
        p_adj = pvals_corrected[i]
        is_significant = reject[i]
        
        # Marcador de significancia basado en el valor p ajustado
        sig_marker = "***" if p_adj < 0.001 else "**" if p_adj < 0.01 else "*" if p_adj < 0.05 else ""
        
        print(f"{v:<52} {n:>5} {f:>8.3f} {p_raw:>8.4f} {p_adj:>8.4f} {sig_marker:>5}")
        
        if is_significant:
            significant_vars.append(r)

    # ── Tukey HSD para variables significativas (basado en el valor p AJUSTADO) ──
    if significant_vars:
        print(f"\n\n{'─'*105}")
        print(f"  TUKEY HSD — Solo variables significativas (FDR Adjusted p < {alpha})")
        print(f"{'─'*105}")
        
        for r in significant_vars:
            v = r['var']
            print(f"\n► Post-hoc para: {v}")
            
            # Volver a ejecutar Tukey usando el subconjunto limpio (que ya solo tiene las 4 orientaciones)
            subset = r['subset']
            res = mc.MultiComparison(subset[v], subset['s1_orientacion_teo'])
            tukey_result = res.tukeyhsd()
            print(tukey_result.summary())
    else:
        print(f"\n\n► Ninguna variable superó la corrección FDR con alpha={alpha}.")

    return raw_results, pvals_corrected

# For Section 2 (Zeros already NaN)
results_s2 = run_anova_per_variable_corrected(df, variables_s2, "Section 2", exclude_zeros=False)

# For Section 3 (Zeros already NaN)
results_s3 = run_anova_per_variable_corrected(df, variables_s3, "Section 3", exclude_zeros=False)

# For Section 4 (Remove zeroes dynamically)
results_s4 = run_anova_per_variable_corrected(df, variables_s4, "Section 4", exclude_zeros=True)


  ANOVA & FDR CORRECTION — Section 2 (ceros incluidos)

Variable                                                 N        F    Raw p    Adj p   sig
-----------------------------------------------------------------------------------------------
s2_evidencia_cientifica                                256   51.541   0.0000   0.0000   ***
s2_experiencia_personal                                252    2.393   0.0690   0.1035      
s2_entrenamiento_clinica                               253    3.455   0.0171   0.0343     *
s2_tratamiento_preferencia_consultantes                233    1.488   0.2186   0.2186      
s2_intuicion                                           213    1.728   0.1623   0.1948      
s2_terapia_personal                                    239   17.291   0.0000   0.0000   ***


─────────────────────────────────────────────────────────────────────────────────────────────────────────
  TUKEY HSD — Solo variables significativas (FDR Adjusted p < 0.05)
───────────────────────────

In [42]:
import numpy as np
import scipy.stats as stats
import statsmodels.stats.multicomp as mc
import statsmodels.stats.multitest as smt

def run_anova_per_variable_bonferroni(df, variables, section_name, exclude_zeros=True, alpha=0.05):
    """
    Ejecuta un ANOVA unidireccional por variable con exclusión de ceros y corrección BONFERRONI.
    Solo incluye 4 orientaciones teóricas específicas.
    Seguido de Tukey HSD solo para las variables significativas.
    """
    
    # Definir los 4 grupos que queremos analizar
    subset_orientations = [
        'Ecléctico (más de una de estas opciones)',
        'Terapias Cognitivas/Comportamentales',
        'Psicoanálisis',
        'Sistémica'
    ]
    
    print(f"\n{'='*105}")
    print(f"  ANOVA & BONFERRONI CORRECTION — {section_name} ({'ceros excluidos' if exclude_zeros else 'ceros incluidos'})")
    print(f"{'='*105}")
    
    # Almacenar resultados preliminares
    raw_results = []
    
    for v in variables:
        # Crear un subconjunto solo con la variable dependiente y la independiente
        subset = df[['s1_orientacion_teo', v]].copy()
        
        # 🔴 FILTRO CRÍTICO: Mantener solo las 4 orientaciones deseadas
        subset = subset[subset['s1_orientacion_teo'].isin(subset_orientations)]
        
        # Excluir ceros/nulos dinámicamente para esta variable específica
        if exclude_zeros:
            subset = subset[subset[v] != 0]
            
        subset = subset.dropna()
        
        # Agrupar los datos
        groups = [group[v].values for name, group in subset.groupby('s1_orientacion_teo')]
        total_n = sum(len(g) for g in groups)
        
        # Comprobar si tenemos suficientes grupos para ejecutar un ANOVA
        if len(groups) < 2 or any(len(g) == 0 for g in groups):
            raw_results.append({'var': v, 'n': total_n, 'f': np.nan, 'p_raw': np.nan, 'eta2': np.nan, 'groups': groups, 'subset': subset})
            continue
            
        # Ejecutar ANOVA estándar
        f_val, p_val = stats.f_oneway(*groups)
        
        # Calcular Eta-Cuadrado (Tamaño del efecto)
        all_data = np.concatenate(groups)
        grand_mean = np.mean(all_data)
        ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
        ss_total = sum((x - grand_mean)**2 for x in all_data)
        eta2 = ss_between / ss_total if ss_total > 0 else np.nan
        
        raw_results.append({
            'var': v, 
            'n': total_n, 
            'f': f_val, 
            'p_raw': p_val, 
            'eta2': eta2,
            'groups': groups,
            'subset': subset
        })

    # Filtrar las variables que no se pudieron calcular antes de la corrección
    valid_results = [r for r in raw_results if not np.isnan(r['p_raw'])]
    p_values = [r['p_raw'] for r in valid_results]
    
    # 🔴 APLICAR CORRECCIÓN BONFERRONI
    reject, pvals_corrected, _, _ = smt.multipletests(p_values, alpha=alpha, method='bonferroni')
    
    # Imprimir la tabla de resumen
    print(f"\n{'Variable':<52} {'N':>5} {'F':>8} {'Raw p':>8} {'Adj p':>8} {'sig':>5}")
    print("-" * 95)
    
    significant_vars = []
    
    for i, r in enumerate(valid_results):
        v = r['var']
        n = r['n']
        f = r['f']
        p_raw = r['p_raw']
        p_adj = pvals_corrected[i]
        is_significant = reject[i]
        
        # Marcador de significancia basado en el valor p ajustado
        sig_marker = "***" if p_adj < 0.001 else "**" if p_adj < 0.01 else "*" if p_adj < 0.05 else ""
        
        print(f"{v:<52} {n:>5} {f:>8.3f} {p_raw:>8.4f} {p_adj:>8.4f} {sig_marker:>5}")
        
        if is_significant:
            significant_vars.append(r)

    # ── Tukey HSD para variables significativas (basado en el valor p AJUSTADO) ──
    if significant_vars:
        print(f"\n\n{'─'*105}")
        print(f"  TUKEY HSD — Solo variables significativas (Bonferroni Adjusted p < {alpha})")
        print(f"{'─'*105}")
        
        for r in significant_vars:
            v = r['var']
            print(f"\n► Post-hoc para: {v}")
            
            # Volver a ejecutar Tukey usando el subconjunto limpio (que ya solo tiene las 4 orientaciones)
            subset = r['subset']
            res = mc.MultiComparison(subset[v], subset['s1_orientacion_teo'])
            tukey_result = res.tukeyhsd()
            print(tukey_result.summary())
    else:
        print(f"\n\n► Ninguna variable superó la corrección Bonferroni con alpha={alpha}.")

    return raw_results, pvals_corrected

# Ejecutar para las secciones
results_s2_bonf = run_anova_per_variable_bonferroni(df, variables_s2, "Section 2", exclude_zeros=False)
results_s3_bonf = run_anova_per_variable_bonferroni(df, variables_s3, "Section 3", exclude_zeros=False)
results_s4_bonf = run_anova_per_variable_bonferroni(df, variables_s4, "Section 4", exclude_zeros=True)


  ANOVA & BONFERRONI CORRECTION — Section 2 (ceros incluidos)

Variable                                                 N        F    Raw p    Adj p   sig
-----------------------------------------------------------------------------------------------
s2_evidencia_cientifica                                256   51.541   0.0000   0.0000   ***
s2_experiencia_personal                                252    2.393   0.0690   0.4140      
s2_entrenamiento_clinica                               253    3.455   0.0171   0.1028      
s2_tratamiento_preferencia_consultantes                233    1.488   0.2186   1.0000      
s2_intuicion                                           213    1.728   0.1623   0.9738      
s2_terapia_personal                                    239   17.291   0.0000   0.0000   ***


─────────────────────────────────────────────────────────────────────────────────────────────────────────
  TUKEY HSD — Solo variables significativas (Bonferroni Adjusted p < 0.05)
─────────────

## MANCOVA DIAGNOSIS

In [43]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.multivariate.manova import MANOVA
import scipy.stats as stats
import statsmodels.stats.multitest as smt
import statsmodels.stats.multicomp as mc

# ── 1. SAMPLE SIZE DIAGNOSTICS ──────────────────────────────────────────────

def diagnose_missingness(df, variables, section_name):
    """
    Shows exactly how much data is lost under listwise deletion (MANCOVA)
    vs item-wise deletion (ANOVA per item).
    """
    print(f"\n{'='*70}")
    print(f"  MISSINGNESS DIAGNOSTICS — {section_name}")
    print(f"{'='*70}")

    total_n = len(df)
    print(f"\nTotal participants: {total_n}")

    # --- MANCOVA scenario: listwise deletion across all variables + covariates ---
    cols_needed = variables + ['s1_orientacion_teo', 's1_edad', 's1_genero', 
                                's1_horas_semana_pacientes_atendidos']
    df_mancova = df[cols_needed].dropna()
    n_mancova = len(df_mancova)
    pct_lost_mancova = (1 - n_mancova / total_n) * 100

    print(f"\n[MANCOVA — listwise deletion]")
    print(f"  Participants retained : {n_mancova} / {total_n}")
    print(f"  Participants lost     : {total_n - n_mancova} ({pct_lost_mancova:.1f}%)")

    # --- ANOVA scenario: item-wise deletion ---
    print(f"\n[ANOVA per item — item-wise deletion]")
    print(f"  {'Variable':<50} {'N retained':>10} {'N lost':>8} {'% lost':>8}")
    print(f"  {'-'*78}")
    for v in variables:
        cols = [v, 's1_orientacion_teo']
        n_item = df[cols].dropna().__len__()
        pct_lost = (1 - n_item / total_n) * 100
        print(f"  {v:<50} {n_item:>10} {total_n - n_item:>8} {pct_lost:>7.1f}%")

    return df_mancova


# ── 2. MANCOVA (original approach) ──────────────────────────────────────────

def run_mancova(df, variables, section_name):
    """
    Runs the original MANCOVA. Will show how many participants survive
    listwise deletion and whether the model can even be estimated.
    """
    print(f"\n{'='*70}")
    print(f"  MANCOVA — {section_name}")
    print(f"{'='*70}")

    df_clean = df[variables + ['s1_orientacion_teo', 's1_edad', 's1_genero',
                                's1_horas_semana_pacientes_atendidos']].dropna()

    print(f"\n  Participants after listwise deletion: {len(df_clean)}")

    if len(df_clean) < len(variables) + 10:
        print("  ⚠ WARNING: Sample too small to estimate MANCOVA reliably.")
        print("  Skipping model estimation.")
        return None

    try:
        formula = ('s1_orientacion_teo ~ '
                   + ' + '.join(variables)
                   + ' + s1_edad + s1_genero + s1_horas_semana_pacientes_atendidos')

        mancova = MANOVA.from_formula(formula, data=df_clean)
        result = mancova.mv_test()
        print(result)
        return result

    except Exception as e:
        print(f"  ✗ MANCOVA failed: {e}")
        return None


# ── 3. ANOVA PER ITEM + FDR (your working approach) ─────────────────────────

def run_anova_per_variable_corrected(df, variables, section_name, 
                                      exclude_zeros=True, alpha=0.05):
    """
    One-way ANOVA per item with FDR correction and Tukey HSD post-hoc.
    Item-wise deletion preserves maximum sample size.
    """

    subset_orientations = [
        'Ecléctico (más de una de estas opciones)',
        'Terapias Cognitivas/Comportamentales',
        'Psicoanálisis',
        'Sistémica'
    ]

    print(f"\n{'='*105}")
    print(f"  ANOVA & FDR CORRECTION — {section_name} "
          f"({'ceros excluidos' if exclude_zeros else 'ceros incluidos'})")
    print(f"{'='*105}")

    raw_results = []

    for v in variables:
        subset = df[['s1_orientacion_teo', v]].copy()
        subset = subset[subset['s1_orientacion_teo'].isin(subset_orientations)]

        if exclude_zeros:
            subset = subset[subset[v] != 0]

        subset = subset.dropna()

        groups = [group[v].values 
                  for name, group in subset.groupby('s1_orientacion_teo')]
        total_n = sum(len(g) for g in groups)

        if len(groups) < 2 or any(len(g) == 0 for g in groups):
            raw_results.append({
                'var': v, 'n': total_n, 'f': np.nan, 
                'p_raw': np.nan, 'eta2': np.nan, 
                'groups': groups, 'subset': subset
            })
            continue

        f_val, p_val = stats.f_oneway(*groups)

        all_data = np.concatenate(groups)
        grand_mean = np.mean(all_data)
        ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
        ss_total = sum((x - grand_mean)**2 for x in all_data)
        eta2 = ss_between / ss_total if ss_total > 0 else np.nan

        raw_results.append({
            'var': v, 'n': total_n, 'f': f_val,
            'p_raw': p_val, 'eta2': eta2,
            'groups': groups, 'subset': subset
        })

    valid_results = [r for r in raw_results if not np.isnan(r['p_raw'])]
    p_values = [r['p_raw'] for r in valid_results]

    reject, pvals_corrected, _, _ = smt.multipletests(
        p_values, alpha=alpha, method='fdr_bh'
    )

    print(f"\n{'Variable':<52} {'N':>5} {'F':>8} {'Raw p':>8} "
          f"{'Adj p':>8} {'η²':>6} {'sig':>5}")
    print("-" * 100)

    significant_vars = []

    for i, r in enumerate(valid_results):
        p_adj = pvals_corrected[i]
        sig_marker = ("***" if p_adj < 0.001 else 
                      "**"  if p_adj < 0.01  else 
                      "*"   if p_adj < 0.05  else "")

        print(f"{r['var']:<52} {r['n']:>5} {r['f']:>8.3f} "
              f"{r['p_raw']:>8.4f} {p_adj:>8.4f} "
              f"{r['eta2']:>6.3f} {sig_marker:>5}")

        if reject[i]:
            significant_vars.append({**r, 'p_adj': p_adj})

    if significant_vars:
        print(f"\n\n{'─'*105}")
        print(f"  TUKEY HSD — Significant variables (FDR adjusted p < {alpha})")
        print(f"{'─'*105}")

        for r in significant_vars:
            print(f"\n► Post-hoc: {r['var']}  [η² = {r['eta2']:.3f}, "
                  f"adj p = {r['p_adj']:.4f}]")
            res = mc.MultiComparison(r['subset'][r['var']], 
                                      r['subset']['s1_orientacion_teo'])
            print(res.tukeyhsd().summary())
    else:
        print(f"\n► No variable survived FDR correction at alpha={alpha}.")

    return raw_results, pvals_corrected


# ── 4. RUN EVERYTHING & COMPARE ──────────────────────────────────────────────

sections = {
    "Section 2": (df_subset_s2, variables_s2, False),
    "Section 3": (df_subset_s3, variables_s3, False),
    "Section 4": (df_subset_s4, variables_s4, True),
}

for section_name, (df_sec, variables, exclude_zeros) in sections.items():

    # Step 1: show the data loss comparison side by side
    diagnose_missingness(df_sec, variables, section_name)

    # Step 2: attempt MANCOVA (will likely fail or warn on sample size)
    run_mancova(df_sec, variables, section_name)

    # Step 3: run your working ANOVA approach
    run_anova_per_variable_corrected(
        df_sec, variables, section_name, exclude_zeros=exclude_zeros
    )


  MISSINGNESS DIAGNOSTICS — Section 2

Total participants: 261

[MANCOVA — listwise deletion]
  Participants retained : 185 / 261
  Participants lost     : 76 (29.1%)

[ANOVA per item — item-wise deletion]
  Variable                                           N retained   N lost   % lost
  ------------------------------------------------------------------------------
  s2_evidencia_cientifica                                   256        5     1.9%
  s2_experiencia_personal                                   252        9     3.4%
  s2_entrenamiento_clinica                                  253        8     3.1%
  s2_tratamiento_preferencia_consultantes                   233       28    10.7%
  s2_intuicion                                              213       48    18.4%
  s2_terapia_personal                                       239       22     8.4%

  MANCOVA — Section 2

  Participants after listwise deletion: 185
                               Multivariate linear model
             

## Analysis for Proportion of Zeros

In [45]:
from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd

subset_orientations = [
    'Ecléctico (más de una de estas opciones)',
    'Terapias Cognitivas/Comportamentales',
    'Psicoanálisis',
    'Sistémica'
]

short = {
    'Ecléctico (más de una de estas opciones)': 'Ecl',
    'Terapias Cognitivas/Comportamentales':     'TCC',
    'Psicoanálisis':                             'PSA',
    'Sistémica':                                 'Sis'
}

def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def run_proportion_analysis(df, variables, section_name):
    df_sub = df[df['s1_orientacion_teo'].isin(subset_orientations)].copy()

    print(f"\n{'='*105}")
    print(f"  PROPORTION OF NON-RESPONSES (NaN) — {section_name}")
    print(f"{'='*105}")
    print(f"\n{'Variable':<50} {'TCC':>12} {'PSA':>14} {'Ecl':>12} {'Sis':>12} "
          f"{'Total':>12} {'χ²':>7} {'p':>8} {'sig':>5}")
    print("-"*115)

    results = []

    for v in variables:
        if v not in df_sub.columns:
            continue

        contingency = []
        cell_values = {}

        for o, s in short.items():
            grp_all = df_sub[df_sub['s1_orientacion_teo'] == o][v]
            n_total = len(grp_all)
            n_nan = int(grp_all.isna().sum())
            n_valid = n_total - n_nan
            pct = round(100 * n_nan / n_total, 1) if n_total > 0 else 0
            cell_values[s] = f"{n_nan}/{n_total} ({pct}%)"
            contingency.append([n_nan, n_valid])

        ct = np.array(contingency)
        total_nan = int(ct[:, 0].sum())
        total_n = int(ct.sum())
        total_pct = round(100 * total_nan / total_n, 1) if total_n > 0 else 0

        if total_nan == 0:
            print(f"{v:<50} {'—':>12} {'—':>14} {'—':>12} {'—':>12} "
                  f"0/{total_n}(0.0%) {'—':>7} {'—':>8} {'—':>5}")
            results.append({'variable': v, 'total_pct': 0, 'chi2': np.nan,
                            'p': np.nan, 'sig': '—',
                            'TCC_pct': 0, 'PSA_pct': 0,
                            'Ecl_pct': 0, 'Sis_pct': 0})
            continue

        try:
            chi2_val, p_val, dof, expected = chi2_contingency(ct)
            low_expected = (expected < 5).any()
            note = '~' if low_expected else ' '
            sig = sig_label(p_val)
        except Exception:
            chi2_val, p_val, sig, note = np.nan, np.nan, 'err', ' '

        def get_pct(key):
            try:
                return float(cell_values[key].split('(')[1].rstrip('%)').strip())
            except:
                return 0.0

        print(f"{v:<50} {cell_values['TCC']:>12} {cell_values['PSA']:>14} "
              f"{cell_values['Ecl']:>12} {cell_values['Sis']:>12} "
              f"{total_nan}/{total_n}({total_pct}%){note}"
              f"{chi2_val:>6.2f} {p_val:>8.4f} {sig:>5}")

        results.append({
            'variable': v,
            'TCC_pct': get_pct('TCC'),
            'PSA_pct': get_pct('PSA'),
            'Ecl_pct': get_pct('Ecl'),
            'Sis_pct': get_pct('Sis'),
            'total_pct': total_pct,
            'chi2': round(chi2_val, 3) if not np.isnan(chi2_val) else np.nan,
            'p': round(p_val, 4) if not np.isnan(p_val) else np.nan,
            'sig': sig,
            'low_expected': note == '~'
        })

    sig_results = [r for r in results if r.get('p', 1) < 0.05]
    print(f"\n  Significant χ² (p < .05): {len(sig_results)}/{len(results)} variables")
    if sig_results:
        print(f"\n  {'Variable':<50} {'χ²':>7} {'p':>8} {'sig':>5} {'PSA%':>6} {'TCC%':>6}")
        print(f"  {'-'*85}")
        for r in sorted(sig_results, key=lambda x: x['p']):
            print(f"  {r['variable']:<50} {r['chi2']:>7.3f} {r['p']:>8.4f} "
                  f"{r['sig']:>5} {r['PSA_pct']:>5.1f}% {r['TCC_pct']:>5.1f}%")

    print(f"\n  Note: ~ = expected cell frequencies < 5, interpret with caution")
    return pd.DataFrame(results)


# ══════════════════════════════════════════════════════════
# SECTION 2 — use preprocessed.csv (before 0→NaN recoding)
# ══════════════════════════════════════════════════════════
df_raw = pd.read_csv('../../data/preprocessed.csv')
df_s2_prop = df_raw[['s1_orientacion_teo'] + variables_s2].copy()
df_s2_prop = df_s2_prop[df_s2_prop['s1_orientacion_teo'].isin(subset_orientations)]
for v in variables_s2:
    if v in df_s2_prop.columns:
        df_s2_prop[v] = df_s2_prop[v].replace(0, np.nan)

prop_s2 = run_proportion_analysis(df_s2_prop, variables_s2,
    'Section 2 — Factors influencing theoretical orientation')


# ══════════════════════════════════════════════════════════
# SECTIONS 3 & 4 — use featured_data_cluster.csv (0s intact)
# ══════════════════════════════════════════════════════════
df_cluster = pd.read_csv('../../data/featured_data_cluster.csv')

df_s3_prop = df_cluster[['s1_orientacion_teo'] + variables_s3].copy()
df_s4_prop = df_cluster[['s1_orientacion_teo'] + variables_s4].copy()

for v in variables_s3:
    if v in df_s3_prop.columns:
        df_s3_prop[v] = df_s3_prop[v].replace(0, np.nan)
for v in variables_s4:
    if v in df_s4_prop.columns:
        df_s4_prop[v] = df_s4_prop[v].replace(0, np.nan)

prop_s3 = run_proportion_analysis(df_s3_prop, variables_s3,
    'Section 3 — Sources used to improve clinical skills')

prop_s4 = run_proportion_analysis(df_s4_prop, variables_s4,
    'Section 4 — Research attitudes')


# ══════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════
print("\n\n" + "█"*90)
print("  SUMMARY: AVERAGE % NON-RESPONSE BY SECTION AND ORIENTATION")
print("█"*90)
print(f"\n{'Section':<15} {'N vars':>7} {'TCC%':>8} {'PSA%':>10} "
      f"{'Ecl%':>8} {'Sis%':>8} {'Total%':>8}")
print("-"*65)
for label, prop_df in [('Section 2', prop_s2),
                        ('Section 3', prop_s3),
                        ('Section 4', prop_s4)]:
    print(f"{label:<15} {len(prop_df):>7} "
          f"{prop_df['TCC_pct'].mean():>7.1f}% "
          f"{prop_df['PSA_pct'].mean():>9.1f}% "
          f"{prop_df['Ecl_pct'].mean():>7.1f}% "
          f"{prop_df['Sis_pct'].mean():>7.1f}% "
          f"{prop_df['total_pct'].mean():>7.1f}%")


  PROPORTION OF NON-RESPONSES (NaN) — Section 2 — Factors influencing theoretical orientation

Variable                                                    TCC            PSA          Ecl          Sis        Total      χ²        p   sig
-------------------------------------------------------------------------------------------------------------------
s2_evidencia_cientifica                            0/135 (0.0%)    5/80 (6.2%)  0/32 (0.0%)  0/14 (0.0%) 5/261(1.9%)~ 11.53   0.0092    **
s2_experiencia_personal                            6/135 (4.4%)    2/80 (2.5%)  1/32 (3.1%)  0/14 (0.0%) 9/261(3.4%)~  1.13   0.7702    ns
s2_entrenamiento_clinica                           4/135 (3.0%)    4/80 (5.0%)  0/32 (0.0%)  0/14 (0.0%) 8/261(3.1%)~  2.47   0.4812    ns
s2_tratamiento_preferencia_consultantes            16/135 (11.9%)   9/80 (11.2%)  3/32 (9.4%)  0/14 (0.0%) 28/261(10.7%)~  1.94   0.5840    ns
s2_intuicion                                       26/135 (19.3%)  17/80 (21.2%) 5/32 (